# Advanced 05 — Strong Laws, Central Limit Theorems, and LIL

Practice notebook for the **"Strong Laws, Central Limit Theorems, and LIL"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Kolmogorov's Strong Law of Large Numbers

Let \((X_n)\) be independent with \(E[X_n]=0\) and
\(\sum_{n=1}^\infty \operatorname{Var}(X_n)/n^2 < \infty\). Then

$$
\frac{1}{n}\sum_{k=1}^n X_k \to} 0.
$$

For the i.i.d. case with \(E[|X_1|]<\infty\) the classical strong law reads

$$
\bar{X}_n = \frac{1}{n}\sum_{k=1}^n X_k \to} E[X_1].
$$

The convergence is *almost sure* (a.s.): for almost every sample path
\(\bar{X}_n(\omega)\to E[X_1]\), not just in distribution. The plot below
draws many independent paths of \(\bar{X}_n\) for i.i.d. variables with
finite variance — every path eventually glues itself to the population mean.

**Your turn:** change the underlying distribution (e.g. Laplace, Bernoulli,
Student-t with \(\nu>2\)) and confirm the running mean still converges;
then try a heavy-tailed Cauchy and watch the SLLN fail (no finite first moment).


In [ ]:
# SLLN: running mean of i.i.d. variables with finite variance -> a.s. convergence
rng = np.random.default_rng(7)

n = 2000          # horizon
n_paths = 25      # number of independent sample paths
mu_true = 1.5     # population mean (finite variance => SLLN applies)
sigma_true = 2.0

# Draw an (n_paths, n) array of i.i.d. Normal(mu, sigma^2) variables
X = rng.normal(loc=mu_true, scale=sigma_true, size=(n_paths, n))
bar_X = np.cumsum(X, axis=1) / np.arange(1, n + 1)

fig, ax = plt.subplots(figsize=(9, 5))
for j in range(n_paths):
    ax.plot(bar_X[j], color="steelblue", alpha=0.45, lw=0.8)
ax.axhline(mu_true, color="crimson", lw=2, label=f"$E[X_1]=\\mu={mu_true}$")
ax.set_title("Kolmogorov SLLN: $\\bar{X}_n \\to \\mu$  (i.i.d., finite variance)")
ax.set_xlabel("$n$")
ax.set_ylabel("$\\bar{X}_n$")
ax.legend()
plt.tight_layout()
plt.show()

# Quantitative check: how close is the last point of each path to mu?
final_err = np.abs(bar_X[:, -1] - mu_true)
print(f"max |bar_X_n - mu| over {n_paths} paths: {final_err.max():.4f}")
print(f"mean |bar_X_n - mu|: {final_err.mean():.4f}")


---
## Part 2 — Lindeberg–Feller CLT (triangular array)

A triangular array \(X_{n,k}\), \(1\leq k\leq k_n\), of independent
mean-zero variables with variances \(\sigma_{n,k}^2\) satisfies
\(s_n^2 = \sum_k \sigma_{n,k}^2 \to \sigma^2>0\). If **Lindeberg's
condition** holds,

$$
\forall \varepsilon>0,\quad
\lim_{n\to\infty} \frac{1}{s_n^2}\sum_{k=1}^{k_n}
E\!\left[X_{n,k}^2 \mathbf{1}_{\{|X_{n,k}|>\varepsilon s_n\}}\right] = 0,
$$

then

$$
\frac{1}{s_n}\sum_{k=1}^{k_n} X_{n,k} \Rightarrow N(0,\sigma^2).
$$

The condition says: no single term carries a non-negligible fraction of the
total variance, and large jumps are negligible in \(L^2\). We build a
triangular array of *non-identically* distributed bounded variables
(uniform on \([-a_{n,k}, a_{n,k}]\) with varying widths), standardize the
row sum by \(s_n\), and check (i) Lindeberg's condition numerically and
(ii) that the standardized row sums are close to \(N(0,\sigma^2)\).

**Your turn:** make one row's largest variance dominate (e.g. set
\(\sigma_{n,1}^2 = 0.9\,s_n^2\)) and watch Lindeberg's condition blow up —
the distribution of the standardized sum is then driven by a single term and
departs from Gaussian.


In [ ]:
# Lindeberg-Feller: triangular array of non-identical bounded variables
rng = np.random.default_rng(11)

n_rows = 5000          # number of rows in the triangular array (Monte Carlo reps)
k_n = 30               # columns per row
sigma2_target = 1.0     # sigma^2 limit

# Row-specific variances: spread the k_n variances so none dominates
weights = np.arange(1, k_n + 1, dtype=float)
weights = weights / weights.sum() * sigma2_target          # sigma_{n,k}^2
sigmas = np.sqrt(weights)
# Variables: Uniform[-a, a] has variance a^2/3, so a = sqrt(3)*sigma
a_s = np.sqrt(3.0) * sigmas

# Draw the triangular array: shape (n_rows, k_n)
U = rng.uniform(-a_s, a_s, size=(n_rows, k_n))            # mean zero by symmetry
row_sums = U.sum(axis=1)
s_n = np.sqrt((sigmas**2).sum())                          # ~ sigma
Z = row_sums / s_n                                        # standardized row sum

# --- Numerical Lindeberg check for several epsilon ---
for eps in (0.1, 0.3, 0.5):
    indicator = (np.abs(U) > eps * s_n).astype(float)
    lindeberg = (U**2 * indicator).sum(axis=1).mean() / (s_n**2)
    print(f"eps={eps:.2f}: Lindeberg term ~= {lindeberg:.5f} (should -> 0)")

# --- Compare standardized row sums to N(0, sigma^2) ---
xs = np.linspace(-4, 4, 400)
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(Z, bins=60, density=True, color="steelblue", alpha=0.6,
        label=f"standardized row sums (k_n={k_n})")
ax.plot(xs, stats.norm.pdf(xs, 0, np.sqrt(sigma2_target)),
        color="crimson", lw=2, label=f"$N(0, \\sigma^2={sigma2_target})$")
ax.set_title("Lindeberg-Feller CLT: standardized sums of non-i.i.d. bounded variables")
ax.set_xlabel("$Z$")
ax.set_ylabel("density")
ax.legend()
plt.tight_layout()
plt.show()

# Goodness of fit: Kolmogorov-Smirnov against N(0, sigma^2)
ks_stat, ks_p = stats.kstest(Z, "norm", args=(0, np.sqrt(sigma2_target)))
print(f"KS stat={ks_stat:.4f}, p-value={ks_p:.3f}  (high p => Gaussian fit is good)")


---
## Part 3 — Law of the Iterated Logarithm (LIL)

For i.i.d. mean-zero variance-one \(X_n\) with partial sums
\(S_n=\sum_{k=1}^n X_k\), the LIL says that almost surely

$$
\limsup_{n\to\infty} \frac{S_n}{\sqrt{2n\log\log n}} = 1,
\quad
\liminf_{n\to\infty} \frac{S_n}{\sqrt{2n\log\log n}} = -1.
$$

So \(\sqrt{2n\log\log n}\) is the *ultimate envelope* of the
fluctuations of \(S_n\): the path is \(O(\sqrt{n})\), but the precise
oscillating scale \(\sqrt{2n\log\log n}\) is hit infinitely often from
both sides. Below we plot \(S_n\) for standard normals against the
envelopes \(\pm\sqrt{2n\log\log n}\).

**Your turn:** generate several paths and check that almost every one
crosses the \(+1\) and \(-1\) normalized envelopes infinitely often as
\(n\) grows; then replace \(X_n\) by \(\pm 1\) Rademacher steps and
re-run — the LIL envelope is universal (it only needs mean 0, variance 1).


In [ ]:
# LIL: partial sums of standard normals vs sqrt(2 n log log n) envelopes
rng = np.random.default_rng(3)

n = 200_000
X = rng.standard_normal(n)
S = np.cumsum(X)

nn = np.arange(3, n + 1)                      # log(log(n)) > 0 only for n >= 3
envelope = np.sqrt(2.0 * nn * np.log(np.log(nn)))
S_pos = S[2:]                                  # S_3 ... S_n  (length n-2)

# Normalized path: S_n / sqrt(2 n log log n)
normalized = S_pos / envelope

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: raw partial sums with envelopes
axes[0].plot(nn, S_pos, color="steelblue", lw=0.5, label="$S_n$")
axes[0].plot(nn, envelope, color="crimson", lw=1.5,
             label="$+\\sqrt{2n\\,\\log\\log n}$")
axes[0].plot(nn, -envelope, color="crimson", lw=1.5,
             label="$-\\sqrt{2n\\,\\log\\log n}$")
axes[0].set_title("LIL: partial sums $S_n$ and the $\\sqrt{2n\\,\\log\\log n}$ envelopes")
axes[0].set_xlabel("$n$")
axes[0].set_ylabel("$S_n$")
axes[0].legend(loc="upper left", fontsize=8)

# Right: normalized path — limsup -> +1, liminf -> -1
axes[1].plot(nn, normalized, color="steelblue", lw=0.5,
             label="$S_n / \\sqrt{2n\\,\\log\\log n}$")
axes[1].axhline(1.0, color="crimson", lw=1.5, linestyle="--", label="$+1$ (limsup)")
axes[1].axhline(-1.0, color="crimson", lw=1.5, linestyle="--", label="$-1$ (liminf)")
axes[1].set_title("Normalized LIL envelope: limsup = +1, liminf = -1")
axes[1].set_xlabel("$n$")
axes[1].set_ylim(-3, 3)
axes[1].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

print(f"Final normalized value:        {normalized[-1]:.4f}")
print(f"Running max of normalized path: {np.max(normalized):.4f}")
print(f"Running min of normalized path: {np.min(normalized):.4f}")
print(f"(For large n these should be hugging [-1, +1].)")


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. State Kolmogorov's SLLN for an i.i.d. sequence and explain why the
    condition \(E[|X_1|]<\infty\) cannot simply be dropped (give a
    counterexample).

2. Write down Lindeberg's condition for a triangular array and explain
    in one sentence what it rules out. Verify numerically that it holds for
    the uniform-bounded array built in Part 2.

3. State the Law of the Iterated Logarithm and explain why its envelope
    \(\sqrt{2n\log\log n}\) is tighter than the CLT scale \(\sqrt{n}\)
    when describing the *extreme* almost-sure fluctuations of \(S_n\).

4. Simulate 1000 paths of \(\bar{X}_n\) for i.i.d. Bernoulli(1/2)
    variables up to \(n=5000\). What fraction of paths are within
    \(\pm 0.02\) of \(1/2\) at \(n=5000\)? Does this fraction grow with
    \(n\)?

5. For the LIL plot, replace the standard-normal increments by Rademacher
    steps \(X_n\in\{-1,+1\}\) with probability \(1/2\) each. Re-plot
    the normalized envelope. Does the LIL still appear to hold? Why?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** For i.i.d. \(X_n\) with \(E[|X_1|]<\infty\),

$$
\bar{X}_n=\frac{1}{n}\sum_{k=1}^n X_k \to} E[X_1].
$$

The integrability condition is essential: take \(X_n\sim\) Cauchy. The
Cauchy has no finite mean, \(\bar{X}_n\) has the *same* Cauchy distribution
for every \(n\) (stable law with index 1), so it cannot converge to a
constant. The running mean wanders forever and the SLLN fails.

**2.** For row \(n\), with \(s_n^2=\sum_k \sigma_{n,k}^2\),

$$
\forall \varepsilon>0,\quad
\frac{1}{s_n^2}\sum_{k=1}^{k_n}
E\!\left[X_{n,k}^2 \mathbf{1}_{\{|X_{n,k}|>\varepsilon s_n\}}\right]
\to 0.
$$

It rules out a single term dominating the variance / making large jumps
that contribute non-negligibly to \(s_n^2\). In Part 2 each
\(X_{n,k}\sim\mathrm{Uniform}[-a_{n,k},a_{n,k}]\) is bounded by
\(a_{n,k}=\sqrt{3}\,\sigma_{n,k}\), and since
\(\max_k \sigma_{n,k}^2 / s_n^2 \to 0\) as \(k_n\to\infty\), for any
fixed \(\varepsilon>0\) the indicator is eventually zero for all terms and
Lindeberg's condition is satisfied — the printed values are essentially 0.

**3.** LIL: for i.i.d. mean-zero variance-one \(X_n\),

$$
\limsup_{n\to\infty}\frac{S_n}{\sqrt{2n\log\log n}}=1,\qquad
\liminf_{n\to\infty}\frac{S_n}{\sqrt{2n\log\log n}}=-1.
$$

The CLT says a *single* time-\(n\) snapshot scaled by \(\sqrt{n}\) is
\(N(0,1)\); it is a distributional statement at one \(n\). The LIL is a
pathwise statement across *all* \(n\): the largest and smallest
fluctuations ever reached are governed by the slightly larger scale
\(\sqrt{2n\log\log n}\). The extra \(\sqrt{2\log\log n}\) factor is
exactly what is needed to capture the extreme almost-sure oscillations —
\(\sqrt{n}\) alone is too small to bound \(\limsup S_n\) (which is
\(+\infty\) a.s.) and too large to describe the precise limsup value.

**4.** Code sketch:

```python
rng = np.random.default_rng(2024)
n_paths, n = 1000, 5000
B = rng.binomial(1, 0.5, size=(n_paths, n))
bar = np.cumsum(B, axis=1) / np.arange(1, n + 1)
within_5000 = np.mean(np.abs(bar[:, -1] - 0.5) <= 0.02)
print(within_5000)   # ~0.6-ish; re-run at n=20000 to see it climb toward 1
```

As \(n\) grows the SLLN forces every path into any fixed-width band around
\(1/2\), so the fraction inside \(\pm 0.02\) monotonically creeps toward
1 (here limited by \(n=5000\) and the \(0.02\) tolerance).

**5.** Yes. The LIL only needs mean 0 and variance 1 (plus finite second
moment / i.i.d.); the distribution shape is irrelevant. Rademacher steps have
\(E[X_n]=0\), \(\operatorname{Var}(X_n)=1\), so the identical envelope
\(\pm\sqrt{2n\log\log n}\) still applies and the normalized path still
has limsup \(+1\), liminf \(-1\). Re-running the Part 3 code with
`X = rng.integers(0, 2, size=n) * 2 - 1` gives a visually identical envelope
hug (with slightly choppier increments since the step is discrete).

</details>
